# 배당수익률과 수익성의 관계 — AAPL 파일럿

**연구 질문:** 기업의 배당수익률(dividend yield)과 수익성(ROA)은 시간에 따라 어떤 관계를 보이는가?

**설계:** AAPL(permno 14593) 단일 종목으로 지표 산출 파이프라인을 검증하는 파일럿.
"관계"의 일반화는 크로스섹션(예: S&P500 전 종목) 확장이 필요하며, 이 노트북은 그 첫 단계다.

**지표 정의:**
- **연간 배당수익률** = 해당 연도 주당 배당 합계 ÷ 연말 수정주가
  - 초기 버전은 배당 이벤트별 `divamt/prc`의 단순평균을 썼는데, 이는 연환산 수익률이
    아니어서(분기배당 종목이면 실제의 ~1/4 스케일) 폐기했다.
- **ROA** = 당기순이익(ni) ÷ 총자산(at), Compustat 연간 재무제표 기준

**데이터 (WRDS):** `crsp.dse`(배당 이벤트), `crsp.dsf`(일별 주가), `comp.funda`(연간 재무),
`crsp.ccmxpf_linktable`(permno↔gvkey).

> **실행에는 WRDS 학술 계정이 필요하다** (`WRDS_USERNAME` 환경변수 설정).

In [ ]:
import os
import wrds
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

db = wrds.Connection(wrds_username=os.environ["WRDS_USERNAME"])
PERMNO = 14593  # AAPL
print("connected")

## 1. 연간 배당 합계와 연말 수정주가

In [ ]:
divs = db.raw_sql("""
    SELECT EXTRACT(YEAR FROM paydt)::int AS year,
           SUM(divamt) AS total_div
    FROM crsp.dse
    WHERE permno = %(permno)s
      AND divamt IS NOT NULL
      AND paydt >= '2012-01-01'
    GROUP BY 1
    ORDER BY 1
""", params={"permno": PERMNO})

px = db.raw_sql("""
    SELECT date, ABS(prc) / NULLIF(cfacpr, 0) AS adj_prc,
           ABS(prc) AS raw_prc
    FROM crsp.dsf
    WHERE permno = %(permno)s
      AND date >= '2012-01-01'
    ORDER BY date
""", params={"permno": PERMNO}, date_cols=["date"])

year_end = (px.assign(year=px["date"].dt.year)
              .groupby("year").last()[["raw_prc"]]
              .rename(columns={"raw_prc": "year_end_prc"}))

div_yield = divs.merge(year_end, on="year")
# 같은 해 안에서는 divamt와 연말 prc가 같은 주식수 기준이라 원시가격으로 나눈다
# (연중 분할이 있으면 미세 오차 — 한계 섹션 참고)
div_yield["div_yield_pct"] = div_yield["total_div"] / div_yield["year_end_prc"] * 100
div_yield

## 2. 연간 수익성 (ROA)

In [ ]:
roa = db.raw_sql("""
    WITH link AS (
        SELECT gvkey, lpermno AS permno,
               linkdt, COALESCE(linkenddt, CURRENT_DATE) AS linkenddt
        FROM crsp.ccmxpf_linktable
        WHERE linktype IN ('LU', 'LC')
          AND linkprim IN ('P', 'C')
          AND lpermno = %(permno)s
    )
    SELECT EXTRACT(YEAR FROM f.datadate)::int AS year,
           f.ni / NULLIF(f.at, 0) AS roa
    FROM comp.funda f
    JOIN link l
      ON f.gvkey = l.gvkey
     AND f.datadate BETWEEN l.linkdt AND l.linkenddt
    WHERE f.indfmt = 'INDL'
      AND f.datafmt = 'STD'
      AND f.consol  = 'C'
      AND f.popsrc  = 'D'
      AND f.datadate >= '2012-01-01'
    ORDER BY 1
""", params={"permno": PERMNO})
roa

## 3. 두 지표의 시계열 비교

In [ ]:
merged = div_yield.merge(roa, on="year").dropna()

fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.bar(merged["year"], merged["div_yield_pct"], color="#1f77b4", alpha=0.6,
        label="dividend yield (%)")
ax1.set_ylabel("dividend yield (%)", color="#1f77b4")
ax2 = ax1.twinx()
ax2.plot(merged["year"], merged["roa"] * 100, color="#d62728", marker="o",
         linewidth=2, label="ROA (%)")
ax2.set_ylabel("ROA (%)", color="#d62728")
ax1.set_title("AAPL: dividend yield vs ROA")
ax1.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("aapl_divyield_vs_roa.png", dpi=120)
plt.show()

corr = merged["div_yield_pct"].corr(merged["roa"])
print(f"연도별 상관계수: {corr:.3f}  (n={len(merged)})")

## 해석 가이드 및 한계

**보는 법:** AAPL은 성장기(高ROA)에 배당수익률이 낮고, 성숙기로 갈수록 주주환원이 커지는
전형적 라이프사이클 가설을 검증하는 사례다. 상관계수의 부호가 그 방향을 요약한다.

**한계:**
1. **n이 10 남짓한 단일 종목** — 통계적 결론 불가. 크로스섹션(연도×종목 패널)으로 확장해
   산업 고정효과를 넣는 것이 다음 단계.
2. 연중 주식분할이 있으면 배당(분할 전 지급분)과 연말 주가의 주식수 기준이 어긋난다 —
   엄밀하게는 배당도 지급일 기준 `cfacpr`로 조정해야 한다.
3. 자사주 매입이 배당을 대체하는 기업(AAPL 포함)에서는 배당수익률이 주주환원 전체를
   과소평가한다 — total payout yield로의 확장이 필요하다.